# Databricks SQL Week - Day 3
## Joins, Set Operations and Multi-Table Query Patterns

**Date:** 27 May 2026

---

## Topics Covered Today

| # | Topic |
|---|-------|
| 1 | Why Joins Are Important |
| 2 | INNER JOIN |
| 3 | LEFT JOIN |
| 4 | RIGHT JOIN |
| 5 | FULL OUTER JOIN |
| 6 | CROSS JOIN |
| 7 | SELF JOIN |
| 8 | Multi-Condition JOIN |
| 9 | Semi-Joins (EXISTS) |
| 10 | Anti-Joins (NOT EXISTS) |
| 11 | Join Pitfalls - Fanout |
| 12 | Set Operations (UNION ALL, UNION, INTERSECT, EXCEPT) |
| 13 | Date Spine Pattern |

---

## Schema Setup

We will use the tables created in the **Setup Notebook**:

| Table | Key Columns |
|-------|-------------|
| `users` | id, email, first_name, last_name, role, phone |
| `orders` | id, user_id, product_id, quantity, unit_price, total_amount, status |
| `products` | id, category_id, name, price, sku, is_active |
| `categories` | id, name, slug, parent_id |
| `inventory` | id, product_id, quantity, reorder_level |

In [0]:
USE CATALOG workspace;
USE SCHEMA ecommerce_sql;

SHOW TABLES;

---

# 1. Why Joins Are Important

In relational databases, data is split across multiple tables. Joins let us combine data from two or more tables based on a related column.

### Primary Key and Foreign Key

| Concept | Definition | Example |
|---------|------------|---------|
| Primary Key (PK) | A unique identifier for each row in a table | `users.id` |
| Foreign Key (FK) | A column that references the primary key of another table | `orders.user_id` references `users.id` |

### Cardinality

| Relationship | Meaning | Example |
|-------------|---------|--------|
| One-to-One (1:1) | One row in Table A matches exactly one row in Table B | `products.id` to `inventory.product_id` |
| One-to-Many (1:N) | One row in Table A matches many rows in Table B | `users.id` to `orders.user_id` |
| Many-to-Many (N:M) | Many rows match many rows, resolved via a junction table | `users` to `products` via `orders` |

### Alias

An *alias* is a temporary name given to a table or column for the duration of a query. It makes results easier to read and queries easier to write, especially when joining tables or renaming columns in the output. Use the `AS` keyword (optional in most SQL dialects).

**Example:**  
sql
```
SELECT u.first_name AS user_first, o.total_amount AS order_value
FROM users u
JOIN orders o ON u.id = o.user_id;
```

---

# 2. INNER JOIN

An `INNER JOIN` returns rows **only** when there is a matching key in both the left and right tables. If a key exists on the left but not on the right (or vice-versa), that row is completely excluded from the result.

| Left Table (Users) | Right Table (Orders)         | Result      |
|--------------------|-----------------------------|-------------|
| User_ID: 2001      | No Orders Exist             | Excluded    |
| User_ID: 2002      | Order Placed                | Included    |


**Syntax:**
```sql
SELECT columns
FROM table_a
INNER JOIN table_b
    ON table_a.key = table_b.key;
```

---

### Example 2.1 - Basic INNER JOIN between two tables

Get users who have placed orders, along with their order details.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    o.id AS order_id,
    o.total_amount
FROM users u
INNER JOIN orders o 
    ON u.id = o.user_id
LIMIT 10;

### Example 2.2 - INNER JOIN with a WHERE filter

Get users who have placed orders that have been delivered, along with their order details.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    o.id AS order_id,
    o.total_amount
FROM users u
INNER JOIN orders o 
    ON u.id = o.user_id
WHERE o.status = 'delivered'
ORDER BY o.total_amount DESC;

### Example 2.3 - INNER JOIN across three tables

Join users, orders, and products together to see who ordered what.

In [0]:
SELECT 
    u.first_name,
    p.name AS product_name,
    o.quantity,
    o.total_amount
FROM orders o
INNER JOIN users u 
    ON o.user_id = u.id
INNER JOIN products p 
    ON o.product_id = p.id
LIMIT 10;

### Example 2.4 - INNER JOIN between products and inventory

Get product name, price, and stock quantity for products tracked in inventory.

In [0]:
SELECT 
    p.name AS product_name,
    p.price,
    p.sku,
    i.quantity AS stock_qty
FROM products p
INNER JOIN inventory i 
    ON p.id = i.product_id;


### When to Use INNER JOIN

Use an `INNER JOIN` when you want to return only the rows where there is a matching value in both tables. This is ideal when you need data that exists in both sources, such as finding users who have placed orders, or products that have inventory records. If a row is missing from either table, it will not appear in the result.

---

# 3. LEFT JOIN (LEFT OUTER JOIN)

A `LEFT JOIN` returns **all rows** from the left table, and the matching rows from the right table. If there is no match on the right side, the result columns for the right table will contain `NULL` values.

| Left Table Row | Right Table Match? | Result |
|---------------|-------------------|--------|
| User A | Has orders | User A + order data |
| User B | No orders | User B + NULL for order columns |

**Syntax:**
```sql
SELECT columns
FROM table_a
LEFT JOIN table_b
    ON table_a.key = table_b.key;
```

---

### Example 3.1 - Basic LEFT JOIN

Show all users with their orders. Users without orders will have NULL in order columns.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    o.id AS order_id,
    o.total_amount
FROM users u
LEFT JOIN orders o 
    ON u.id = o.user_id
ORDER BY o.id ASC;

### Example 3.2 - LEFT JOIN + WHERE IS NULL (find non-matching rows)

Find users who have never placed an order.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
LEFT JOIN orders o 
    ON u.id = o.user_id
WHERE o.id IS NULL;

### Example 3.3 - LEFT JOIN with products and categories

Show all products with their category name. Products without a category will show NULL.

In [0]:
SELECT 
    p.name AS product_name,
    p.price,
    c.name AS category_name
FROM products p
LEFT JOIN categories c 
    ON p.category_id = c.id
ORDER BY p.price DESC;

### When to Use LEFT JOIN

Use a `LEFT JOIN` when you want to return all rows from the left table, along with any matching rows from the right table. This is useful for finding records in the left table that may not have related entries in the right table. For example, to list all users and show their orders if they have any, or to identify items that are missing related data. If there is no match, the right table's columns will contain `NULL`.

---

# 4. RIGHT JOIN (RIGHT OUTER JOIN)

A `RIGHT JOIN` returns **all rows from the right table**, and the matching rows from the left table. If there is no match on the left side, the result columns for the left table will contain `NULL` values.

| Right Table Row | Left Table Match? | Result |
|-----------------|------------------|--------|
| Order A         | Has user         | Order A + user data |
| Order B         | No user          | Order B + NULL for user columns |

It is the mirror of LEFT JOIN. In practice, most people rewrite RIGHT JOINs as LEFT JOINs by swapping the table order.

| LEFT JOIN | RIGHT JOIN (equivalent) |
|-----------|------------------------|
| `FROM A LEFT JOIN B` | `FROM B RIGHT JOIN A` |
| Keeps all rows from A | Keeps all rows from A |

---

### Example 4.1 - Basic RIGHT JOIN

Show all orders with user info. If an order has no matching user, user columns will be NULL.

In [0]:
SELECT 
    u.first_name,
    u.last_name,
    o.id AS order_id,
    o.total_amount
FROM users u
RIGHT JOIN orders o 
    ON u.id = o.user_id
LIMIT 10;

**When to Use RIGHT JOIN**

Use a `RIGHT JOIN` when you want to return all rows from the right table, along with any matching rows from the left table. This is useful if you need to ensure every row from the right table appears in your results, even if there is no related data in the left table. For example, to list all orders and show user info if available, or to find records in the right table that may not have related entries in the left table. If there is no match, the left table's columns will contain `NULL`.

In practice, `RIGHT JOIN` is less common than `LEFT JOIN` you can usually swap table order and use a `LEFT JOIN` instead.

---

# 5. FULL OUTER JOIN

A `FULL OUTER JOIN` returns **all rows from both tables**. If a row from the left table has no matching row in the right table, the right table's columns will contain `NULL`. If a row from the right table has no matching row in the left table, the left table's columns will contain `NULL`. Only rows that have no match on either side will have `NULL` values for the other table's columns.

| Left Match | Right Match | Result |
|-----------|------------|--------|
| Yes | Yes | Both sides filled |
| Yes | No | Right side is NULL |
| No | Yes | Left side is NULL |

This is useful for data audits and finding orphaned records.

---

### Example 5.1 - Basic FULL OUTER JOIN

Show all users and all orders, even if they dont match.

In [0]:
SELECT 
    u.id AS user_id,
    u.email,
    o.id AS order_id,
    o.total_amount
FROM users u
FULL OUTER JOIN orders o 
    ON u.id = o.user_id;

### Example 5.2 - FULL OUTER JOIN to find orphaned records only

Filter to see only rows where one side is missing.

In [0]:
SELECT 
    u.id AS user_id,
    u.email,
    o.id AS order_id,
    o.total_amount
FROM users u
FULL OUTER JOIN orders o 
    ON u.id = o.user_id
WHERE u.id IS NULL OR o.id IS NULL;

### Example 5.3 - FULL OUTER JOIN between products and inventory

Find products missing from inventory or inventory entries without a valid product.

In [0]:
SELECT 
    p.id AS product_id,
    p.name AS product_name,
    i.id AS inventory_id,
    i.quantity
FROM products p
FULL OUTER JOIN inventory i 
    ON p.id = i.product_id
WHERE i.product_id IS NULL OR p.id IS NULL;

### Join Types Comparison

| Join Type | Left Table Rows Returned | Right Table Rows Returned |
|-----------|------------------------|-------------------------|
| INNER JOIN | Only matching | Only matching |
| LEFT JOIN | All | Only matching (NULL if no match) |
| RIGHT JOIN | Only matching (NULL if no match) | All |
| FULL OUTER JOIN | All (NULL if no match) | All (NULL if no match) |

---

# 6. CROSS JOIN

A `CROSS JOIN` creates a **Cartesian Product**. It matches every single row of Table A with every single row of Table B. If Table A has *X* rows and Table B has *Y* rows, the output will contain *X × Y* rows.

| Table A | Table B | Output      |
|---------|---------|-------------|
| Red     | Circle  | Red Circle  |
| Red     | Square  | Red Square  |
| Blue    | Circle  | Blue Circle |
| Blue    | Square  | Blue Square |


> **Performance Warning:** Running a `CROSS JOIN` on large tables (e.g., 100,000 rows x 100,000 rows) generates 10 billion rows, which can freeze or crash your database cluster. Use carefully and always filter or run on small lists.

**Syntax:**
```sql
SELECT columns
FROM table_a
CROSS JOIN table_b;
```

---

### Example 6.1 - Basic CROSS JOIN

Pair every category with every distinct order status.

In [0]:
SELECT 
    c.name AS category_name,
    s.status
FROM categories c
CROSS JOIN (SELECT DISTINCT status FROM orders) s
ORDER BY c.name, s.status;

### Example 6.2 - CROSS JOIN with a filtered table

Pair admin users with all categories.

In [0]:
SELECT 
    u.first_name,
    u.last_name,
    c.name AS category_name
FROM users u
CROSS JOIN categories c
WHERE u.role = 'admin';

---

# 7. SELF JOIN

A `SELF JOIN` is a standard join where a table is **joined to itself**. This requires referencing the table twice using different aliases (e.g., `Table A` and `Table B`) to avoid catalog lookup collisions.

This is highly used for **hierarchical data** (Employee -> Manager, Category -> Parent Category, or finding duplicate records).

Common use case: hierarchical data where a row references another row in the same table (e.g., `categories.parent_id` pointing to `categories.id`).

---

### Example 7.1 - SELF JOIN to find subcategory and parent

The `categories` table has a `parent_id` column that references the `id` of another row in the same table.

In [0]:
SELECT 
    sub.name AS subcategory_name,
    parent.name AS parent_category_name
FROM categories sub
INNER JOIN categories parent 
    ON sub.parent_id = parent.id
ORDER BY parent_category_name, subcategory_name;

### Example 7.2 - SELF JOIN with LEFT JOIN to include root categories

Root categories have `parent_id = NULL`. Using LEFT JOIN ensures they are not excluded.

In [0]:
SELECT 
    c.name AS category_name,
    parent.name AS parent_name
FROM categories c
LEFT JOIN categories parent 
    ON c.parent_id = parent.id
ORDER BY parent.name NULLS FIRST, c.name;

### Example 7.3 - Show parent categories and their children

Find root categories (parent_id IS NULL) and list their subcategories.

In [0]:
SELECT 
    parent.name AS parent_category,
    child.name AS subcategory
FROM categories parent
LEFT JOIN categories child 
    ON child.parent_id = parent.id
WHERE parent.parent_id IS NULL
ORDER BY parent_category;

---

# 8. Multi-Condition JOIN

You can add multiple conditions in the `ON` clause using `AND`. This is useful when:
- Joining on composite keys
- Adding extra filtering criteria directly in the join

**Syntax:**
```sql
SELECT columns
FROM table_a a
JOIN table_b b
    ON a.id = b.id
   AND a.date BETWEEN b.start AND b.end;
```

---

### Example 8.1 - JOIN with multiple ON conditions

Join orders with products, but only match active products.

In [0]:
SELECT 
    o.id AS order_id,
    p.name AS product_name,
    p.price,
    o.total_amount
FROM orders o
JOIN products p 
    ON o.product_id = p.id
   AND p.is_active = true
LIMIT 10;

### Example 8.2 - JOIN with price threshold in ON clause

Join orders with products where the product price is above 50000.

In [0]:
SELECT 
    o.id AS order_id,
    p.name AS product_name,
    p.price,
    o.quantity
FROM orders o
INNER JOIN products p 
    ON o.product_id = p.id
   AND p.price > 50000.00
WHERE o.status != 'cancelled';

### Example 8.3 - Multi-table JOIN with multiple conditions

Join orders to users and products with extra conditions on each join.

In [0]:
SELECT 
    o.id AS order_id,
    u.first_name,
    p.name AS product_name,
    o.total_amount
FROM orders o
JOIN users u 
    ON o.user_id = u.id
   AND o.shipping_address = o.billing_address
JOIN products p 
    ON o.product_id = p.id
   AND p.is_active = true
WHERE o.status IN ('delivered', 'shipped');

---

# 9. Semi-Joins and Anti-Joins

## Semi-Join using EXISTS

A semi-join checks if a matching row **exists** in another table, but does not add any columns from that table.

| Approach | Duplicates? | Adds columns from right table? |
|----------|------------|-------------------------------|
| INNER JOIN | Yes, if multiple matches | Yes |
| EXISTS | No, always unique | No |

**Syntax:**
```sql
SELECT columns
FROM table_a a
WHERE EXISTS (
    SELECT 1 FROM table_b b WHERE b.key = a.key
);
```

---

### Example 9.1 - EXISTS: Find users who have placed orders

Unlike INNER JOIN, this returns each user only once even if they have many orders.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.user_id = u.id
);

### Example 9.2 - EXISTS with additional filter in the subquery

Find users who have at least one order above 10,000.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name
FROM users u
WHERE EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.user_id = u.id
      AND o.total_amount > 10000
);

### Example 9.3 - EXISTS: Find categories that have products

Returns each category only once, even if it has many products.

In [0]:
SELECT 
    c.id AS category_id,
    c.name AS category_name
FROM categories c
WHERE EXISTS (
    SELECT 1 
    FROM products p 
    WHERE p.category_id = c.id
);

---

# 10. Anti-Joins using NOT EXISTS

An anti-join finds rows in Table A that have **no matching row** in Table B.

### NOT EXISTS vs NOT IN

| Feature | NOT EXISTS | NOT IN |
|---------|-----------|--------|
| NULL-safe | Yes | No - returns 0 rows if any NULL exists in subquery |
| Performance | Stops at first match (fast) | Must check all values |
| Recommended | Yes | Use with caution |

---

In [0]:
-- Example: Find users who have not placed any orders using NOT IN
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE u.id NOT IN (
    SELECT o.user_id 
    FROM orders o
);

### Example 10.1 - NOT EXISTS: Find users with no orders

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE NOT EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.user_id = u.id
);

### Example 10.2 - NOT EXISTS: Find customers with no delivered orders

They might have pending or cancelled orders, but nothing delivered.

In [0]:
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name
FROM users u
WHERE u.role = 'customer'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o 
      WHERE o.user_id = u.id
        AND o.status = 'delivered'
  );

### Example 10.3 - NOT EXISTS: Find products not in inventory

In [0]:
SELECT 
    p.id AS product_id,
    p.name AS product_name,
    p.sku
FROM products p
WHERE NOT EXISTS (
    SELECT 1 
    FROM inventory i 
    WHERE i.product_id = p.id
);

### Example 10.4 - Demonstrating the NOT IN NULL trap

If the subquery returns any NULL value, `NOT IN` returns zero rows.

In [0]:
-- This may return 0 rows if orders.user_id contains any NULL!
SELECT id, first_name 
FROM users 
WHERE id NOT IN (SELECT user_id FROM orders);

In [0]:
-- NOT EXISTS always works correctly regardless of NULLs
SELECT u.id, u.first_name 
FROM users u
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.user_id = u.id
);

In [0]:
SELECT 
    COUNT(*) AS total_orders,
    SUM(total_amount) AS true_revenue
FROM orders;

---

# 12. Set Operations

Joins combine tables **horizontally** (adding columns). Set operations combine query results **vertically** (stacking rows).

| Operation | What it does | Removes duplicates? |
|-----------|-------------|--------------------|
| UNION ALL | Combines all rows from both queries | No |
| UNION | Combines rows and removes duplicates | Yes |
| INTERSECT | Returns only rows found in both queries | Yes |
| EXCEPT | Returns rows from first query not in second | Yes |

**Rule:** Both queries must return the same number of columns with compatible data types.

---

### Example 12.1 - UNION ALL

Combine high-value orders and orders from Maharashtra into one result. Duplicates are kept.

In [0]:
SELECT id AS order_id, user_id, total_amount, 'High Value' AS segment
FROM orders
WHERE total_amount > 50000.00

UNION ALL

SELECT id AS order_id, user_id, total_amount, 'Maharashtra' AS segment
FROM orders
WHERE shipping_address LIKE '%Maharashtra%';

### Example 12.2 - UNION (removes duplicates)

Get unique user IDs from two different groups.

In [0]:
SELECT user_id FROM orders WHERE total_amount > 50000.00

UNION

SELECT user_id FROM orders WHERE status = 'delivered';

### Example 12.3 - INTERSECT

Find user IDs that appear in both groups - users with delivered orders AND users with shipped orders.

In [0]:
SELECT user_id 
FROM orders 
WHERE status = 'delivered'

INTERSECT

SELECT user_id 
FROM orders 
WHERE status = 'shipped';

### Example 12.4 - EXCEPT

Find users who placed orders but never had any delivered.

In [0]:
SELECT user_id FROM orders

EXCEPT

SELECT user_id FROM orders WHERE status = 'delivered';

---

# Day 3 Summary

| Concept | When to Use |
|---------|------------|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | Keep all left rows, NULL for non-matches |
| RIGHT JOIN | Keep all right rows (prefer LEFT JOIN instead) |
| FULL OUTER JOIN | Audit queries, find orphaned records |
| CROSS JOIN | Cartesian product of small tables |
| SELF JOIN | Hierarchical parent-child data |
| Multi-Condition JOIN | Composite keys, extra filters in ON |
| EXISTS | Check if matching row exists (no duplicates) |
| NOT EXISTS | Find rows with no match (NULL-safe) |
| Pre-Aggregate then JOIN | Fix fanout / inflated metrics |
| UNION ALL | Stack rows, keep duplicates |
| UNION | Stack rows, remove duplicates |
| INTERSECT | Rows present in both queries |
| EXCEPT | Rows in first query but not in second |


---

# Practice Challenges

Try these on your own using the workshop tables.

---

### Challenge 1
Write an INNER JOIN to show all orders with the product name and user first name.

In [0]:
-- Challenge 1 Solution
SELECT 
    o.id AS order_id,
    u.first_name AS user_name,
    p.name AS product_name,
    o.quantity,
    o.total_amount
FROM orders o
INNER JOIN users u 
    ON o.user_id = u.id
INNER JOIN products p 
    ON o.product_id = p.id
ORDER BY o.id
LIMIT 20;

### Challenge 2
Use a LEFT JOIN to find all products that have no inventory entry (i.e., inventory columns are NULL).

In [0]:
-- Challenge 2 Solution
SELECT 
    p.id AS product_id,
    p.name AS product_name,
    p.sku,
    p.price
FROM products p
LEFT JOIN inventory i 
    ON p.id = i.product_id
WHERE i.id IS NULL;

### Challenge 3
Use NOT EXISTS to find customers (role = 'customer') who have never placed any order.

In [0]:
-- Challenge 3 Solution
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE u.role = 'customer'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o 
      WHERE o.user_id = u.id
  )
ORDER BY u.id;

### Challenge 4
Use INTERSECT to find user IDs who have both 'delivered' and 'cancelled' orders.

In [0]:
-- Challenge 4 Solution
SELECT user_id 
FROM orders 
WHERE status = 'delivered'

INTERSECT

SELECT user_id 
FROM orders 
WHERE status = 'cancelled';

### Challenge 5
Use EXCEPT to find user IDs who placed orders but never had a 'shipped' order.

In [0]:
-- Challenge 5 Solution
SELECT DISTINCT user_id 
FROM orders

EXCEPT

SELECT DISTINCT user_id 
FROM orders 
WHERE status = 'shipped';

### Challenge 6
Write a self join on the categories table to display each subcategory alongside its parent category name.

In [0]:
-- Challenge 6 Solution
SELECT 
    child.id AS subcategory_id,
    child.name AS subcategory_name,
    parent.id AS parent_id,
    parent.name AS parent_category_name
FROM categories child
INNER JOIN categories parent 
    ON child.parent_id = parent.id
ORDER BY parent.name, child.name;

### Challenge 7
Generate a date spine for the first week of May 2026 and LEFT JOIN daily order counts onto it. Days with no orders should show 0.

In [0]:
-- Challenge 7 Solution
WITH date_spine AS (
    SELECT date_col
    FROM (
        SELECT EXPLODE(
            SEQUENCE(
                DATE '2026-05-01',
                DATE '2026-05-07',
                INTERVAL 1 DAY
            )
        ) AS date_col
    )
),
daily_orders AS (
    SELECT 
        DATE(order_date) AS order_day,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY DATE(order_date)
)
SELECT 
    ds.date_col AS date,
    COALESCE(do.order_count, 0) AS orders
FROM date_spine ds
LEFT JOIN daily_orders do 
    ON ds.date_col = do.order_day
ORDER BY ds.date_col;

### Challenge 8
Write a query that pre-aggregates inventory by product_id, then joins to categories via products to get total stock per category. (Hint: aggregate before joining to avoid fanout)

In [0]:
-- Challenge 8 Solution
WITH inventory_agg AS (
    SELECT 
        product_id,
        SUM(quantity) AS total_quantity
    FROM inventory
    GROUP BY product_id
)
SELECT 
    c.id AS category_id,
    c.name AS category_name,
    SUM(ia.total_quantity) AS total_stock
FROM categories c
INNER JOIN products p 
    ON c.id = p.category_id
INNER JOIN inventory_agg ia 
    ON p.id = ia.product_id
GROUP BY c.id, c.name
ORDER BY total_stock DESC;